In [10]:
import fsspec
import xarray as xr
import pandas as pd 
import os
from datetime import date
import datetime
import regionmask
import geopandas as gpd
import scipy.stats as stats
from ERA5_functions import *

In [7]:
ds = xr.open_mfdataset('/data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc')
t2m = ds.t2m.sel(time=ds.time.dt.year <= 2024)
u = ds.u.sel(time=ds.time.dt.year <= 2024)
v = ds.v.sel(time=ds.time.dt.year <= 2024)

fpath = '/data/keeling/a/rytam2/a/iema_output/arcgis_toprocess/' #for output to arcgis

In [9]:
sfc_wind,_= wind_tot(u,v) # tier 1 
wchill = wind_chill(t2m, sfc_wind) # tier 2

In [ ]:
## tot hours in month 
wchill_f = wchill
hourlymask = wchill_f<-20
grouped_hourly = hourlymask.groupby(['time.year', 'time.month']).sum(dim='time').rename('TOTAL_HOURS_LESS_N20F').stack(time=('year', 'month'))#.to_dataframe().drop(columns=['county', 'year', 'month'])

## tot days in month 
wchill_dmin = wchill_f.resample(time='1D').min(dim='time')
dailymask = wchill_dmin<-20
grouped_daily = dailymask.groupby(['time.year', 'time.month']).sum(dim='time').rename('TOTAL_DAYS_LESS_N20F').stack(time=('year', 'month'))#.to_dataframe().drop(columns=['county', 'year', 'month'])

monthly_compiled = xr.merge([grouped_hourly,grouped_daily]).reset_index('time')
monthly_compiled['time']=pd.date_range(start='2015-01-01', periods=9*12, freq='MS')

# monthly_compiled.to_netcdf(fpath+'monthlystats_'+'windchill'+'_by_county_2016-2024'+'.nc')

In [ ]:
### tot hours in a year (calendar year)
grouped_hourly_yr = hourlymask.groupby(['time.year']).sum(dim='time').rename('TOTAL_HOURS_LESS_N20F')
grouped_daily_yr = dailymask.groupby(['time.year']).sum(dim='time').rename('TOTAL_DAYS_LESS_N20F')

yearly_compiled = xr.merge([grouped_hourly_yr,grouped_daily_yr])

# yearly_compiled.to_netcdf(fpath+'yearlystats_'+'windchill'+'_by_county_2016-2024'+'.nc')

In [ ]:
### summary 
grouped_hourly_smry = hourlymask.sum(dim='time').rename('TOTAL_HOURS_LESS_N20F')#.stack(time=('year', 'month'))
grouped_daily_smry = dailymask.sum(dim='time').rename('TOTAL_DAYS_LESS_N20F')#.stack(time=('year', 'month'))

summary_compiled = xr.merge([grouped_hourly_smry,grouped_daily_smry])
# summary_compiled.to_netcdf(fpath+'summarystats_'+'windchill'+'_by_county_2016-2024'+'.nc')